# Bronze Layer – Data Ingestion

**Overview :** 
This notebook ingests raw World Bank data, performs basic preprocessing, and stores the data as Delta tables in the Bronze layer.

In [0]:
# Environment setup, Schema and Volume
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.global_development")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.global_development.raw_files")

print("Schema and Volume created successfully")

Schema and Volume created successfully


In [0]:
# Display all the files in the raw_files volume
display(dbutils.fs.ls("/Volumes/workspace/global_development/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/global_development/raw_files/_temp_education/,_temp_education/,0,1789114714460
dbfs:/Volumes/workspace/global_development/raw_files/_temp_gdp_growth/,_temp_gdp_growth/,0,1789114714460
dbfs:/Volumes/workspace/global_development/raw_files/_temp_population/,_temp_population/,0,1789114714460
dbfs:/Volumes/workspace/global_development/raw_files/country_metadata.csv,country_metadata.csv,64527,1788613672000
dbfs:/Volumes/workspace/global_development/raw_files/education_expenditure_pct_gdp.csv,education_expenditure_pct_gdp.csv,180444,1788184414000
dbfs:/Volumes/workspace/global_development/raw_files/gdp_growth_annual_pct.csv,gdp_growth_annual_pct.csv,301969,1788184412000
dbfs:/Volumes/workspace/global_development/raw_files/population_total.csv,population_total.csv,195734,1788184413000


**Architecture Raw CSV → Bronze (raw ingestion)**
Ingests raw CSV files from the `raw_files` volume as-is into Bronze Delta tables, with no transformation — preserving the original source data for auditability.


In [0]:
# Set path for all files
raw_path = "/Volumes/workspace/global_development/raw_files/"

gdp_growth_file = raw_path + "gdp_growth_annual_pct.csv"
population_file = raw_path +"population_total.csv"
education_file = raw_path + "education_expenditure_pct_gdp.csv"

In [0]:
# See the structure of the file
df_peek = (
    spark.read
    .option("header",True)
    .option("InferSchema", True)
    .csv(gdp_growth_file)
)

display(df_peek.show(5))

+--------------------+----------------------------+--------------------+
|         Data Source|World Development Indicators|                 _c2|
+--------------------+----------------------------+--------------------+
|   Last Updated Date|                  2026-07-13|                NULL|
|        Country Name|                Country Code|      Indicator Name|
|               Aruba|                         ABW|GDP growth (annua...|
|Africa Eastern an...|                         AFE|GDP growth (annua...|
|         Afghanistan|                         AFG|GDP growth (annua...|
+--------------------+----------------------------+--------------------+
only showing top 5 rows


### Define World Bank Data Schema

This section creates a reusable PySpark schema for World Bank data.
The schema includes country and indicator details along with yearly values from 1960 to 2025.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Build a dynamic schema for World Bank data with country, indicator, and yearly fields.
def build_worldbank_schema(year_start=1960, year_end=2025):
    # Define the fixed metadata fields.
    fields = [
        StructField("Country_Name", StringType(), True),
        StructField("Country_Code", StringType(), True),
        StructField("Indicator_Name", StringType(), True),
        StructField("Indicator_Code", StringType(), True),
    ]
    
    # Add a DoubleType field for each year in the specified range.
    for year in range(year_start, year_end + 1):
        fields.append(StructField(str(year), DoubleType(), True))
    return StructType(fields)

# Create the World Bank schema using the default year range.
worldbank_schema = build_worldbank_schema()

### Load and Prepare World Bank CSV Files

This function reads the raw World Bank CSV files, removes metadata rows, applies the predefined schema, and stores the cleaned data in temporary storage before loading it into Spark DataFrames.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Read, clean, and load a World Bank CSV file into a Spark DataFrame.
def load_worldbank_csv(file_path, temp_folder_name):
    try:
        raw_lines = spark.read.text(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Could not read source file at {file_path}: {e}")
    
    # Add a unique index to identify the position of each raw line.
    indexed_lines = raw_lines.withColumn("idx", monotonically_increasing_id())

    # Remove the first four metadata rows and keep only the data values.
    clean_lines = indexed_lines.filter(indexed_lines.idx >= 4).select("value")
    
    # Define the temporary storage path for the cleaned file.
    temp_path = f"/Volumes/workspace/global_development/raw_files/{temp_folder_name}/"
    clean_lines.write.mode("overwrite").text(temp_path)
    
    # Read the cleaned file using the predefined World Bank schema.
    df = (
        spark.read
        .option("header", True)
        .schema(worldbank_schema)
        .csv(temp_path)
    )
    
    # Replace spaces in column names with underscores for easier processing.
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))
    
    return df
# Load GDP, Population, Education data.
df_gdp_growth = load_worldbank_csv(gdp_growth_file, "_temp_gdp_growth")
df_population = load_worldbank_csv(population_file, "_temp_population")
df_education = load_worldbank_csv(education_file, "_temp_education")

df_gdp_growth.show(3)
df_population.show(3)
df_education.show(3)

+--------------------+------------+--------------------+-----------------+----+----------------+----------------+----------------+----------------+--------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+-----------------+------------------+----------------+-------------------+---------------+----------------+----------------+----------------+-----------------+------------------+-----------------+------------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+------------------+----------------+----------------+------------------+----------------+----------------+----------------+-----------------+-----------------+-----------------+-----------------+----------------+---

In [0]:
# Write the processed DataFrames as Delta tables in the Bronze layer.
df_gdp_growth.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.bronze_gdp_growth")
df_population.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.bronze_population")
df_education.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.bronze_education")

print("Bronze tables created: bronze_gdp_growth, bronze_population, bronze_education")

Bronze tables created: bronze_gdp_growth, bronze_population, bronze_education


In [0]:
display(spark.sql("SHOW TABLES IN workspace.global_development"))

database,tableName,isTemporary
global_development,bronze_country_metadata,false
global_development,bronze_education,false
global_development,bronze_gdp_growth,false
global_development,bronze_population,false
global_development,gold_country_development_metrics,false
global_development,silver_education,false
global_development,silver_education_with_nulls,false
global_development,silver_gdp_growth,false
global_development,silver_gdp_growth_with_nulls,false
global_development,silver_population,false


  # Note: all 3 datasets shipped identical Metadata_Country files (verified separately); 
  # consolidated to a single country_metadata.csv used across all layers.

In [0]:
# Load country metadata (reference/dimension data - simple structure, no junk rows)
df_country_metadata = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/global_development/raw_files/country_metadata.csv")
    .withColumnRenamed("Country Code", "Country_Code")
    .select("Country_Code", "Region", "IncomeGroup")
)

df_country_metadata.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.global_development.bronze_country_metadata"
)

print("Bronze table created: bronze_country_metadata")

Bronze table created: bronze_country_metadata


In [0]:
display(spark.sql("SHOW TABLES IN workspace.global_development"))

database,tableName,isTemporary
global_development,bronze_country_metadata,false
global_development,bronze_education,false
global_development,bronze_gdp_growth,false
global_development,bronze_population,false
global_development,gold_country_development_metrics,false
global_development,silver_education,false
global_development,silver_education_with_nulls,false
global_development,silver_gdp_growth,false
global_development,silver_gdp_growth_with_nulls,false
global_development,silver_population,false
